# P1.01 · Feature pipeline
**End-to-end credit decision engine** · **L3** tier

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1thirteeng3/ds-ai-cookbook/blob/colab/en/projects/P1-credit/P1.01-e2e-credit-decision-engine/notebooks/30-feature-pipeline.ipynb)

| Level | Colab runtime | Time (smoke / full) | Document |
|---|---|---|---|
| 🟠 L3 | cpu | 2 / 8 min | [index.en.md](../index.en.md) |

> Turns application + bureau history into a versioned feature table, with a data contract and a frozen category vocabulary. The same code runs in training and in the API (notebook 33).

## 0. Parameters

`RUN_MODE="smoke"` uses 10% of the data (or the synthetic twin with `OFFLINE=True`) and runs in minutes. `RUN_MODE="full"` uses the whole dataset and produces the official results. `OFFLINE=True` exists only for tests and CI: never draw conclusions from it.

In [ ]:
# Parameters (papermill overrides this cell)
RUN_MODE = "smoke"   # "smoke" | "full"
SEED = 42
DATA_DIR = "data"    # Colab + Drive: "/content/drive/MyDrive/cookbook-data"
OFFLINE = False      # True: synthetic twin (tests/CI). Never for full results.
SAMPLE_FRAC = 0.1    # smoke only
N_BOOT = 200         # bootstrap resamples for metric uncertainty

## 1. Setup

On Colab, installs the `cookbook` package from the private repo (`GH_TOKEN` secret) and the exact versions used for the results. NumPy, pandas and scikit-learn keep Colab's versions and are recorded in the result file. If the install upgrades any of them, use *Runtime > Restart session* and run from section 0. To download Home Credit, create the `KAGGLE_API_TOKEN` secret (or `KAGGLE_USERNAME` and `KAGGLE_KEY`) and accept the competition rules on kaggle.com with the same account.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = {
    "optbinning": "1.0.0",
    "lightgbm": "4.7.0",
    "optuna": "5.0.0",
    "mlflow": "3.18.0",
    "fastapi": "0.162.0",
    "httpx": "0.28.1",
    "polars": "2.1.0",
    "pyarrow": "26.0.1",
    "kaggle": "2.2.5",
    "joblib": "1.6.0",
    "matplotlib": "3.11.2",
}

if IN_COLAB and not sys.modules.get("cookbook"):
    from google.colab import userdata

    token = userdata.get("GH_TOKEN")
    pins = " ".join(f"{k}=={v}" for k, v in PACKAGES.items())
    %pip install -q uv
    !uv pip install --system -q {pins} "cookbook[credit,kaggle] @ git+https://{token}@github.com/1thirteeng3/cookbokk.git@main"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from cookbook import credit as cr
from cookbook import home_credit as hc
from cookbook.data import sample
from cookbook.env import colab_secrets_to_env, hardware_info, quiet_logs, set_seeds, versions
from cookbook.results import Timer, save_result

quiet_logs()  # before heavy imports: silences optional-solver probing
import inspect
import json

import lightgbm as lgb

set_seeds(SEED)
colab_secrets_to_env()
if RUN_MODE == "full" and OFFLINE:
    raise ValueError("RUN_MODE='full' needs real data (OFFLINE=False).")

SOURCE = "home-credit-synthetic" if OFFLINE else "home-credit"
ARTIFACTS = Path(DATA_DIR) / "artifacts" / "P1.01" / RUN_MODE
ARTIFACTS.mkdir(parents=True, exist_ok=True)
HW = hardware_info()
VERSIONS = versions([*PACKAGES, "numpy", "pandas", "scikit-learn"])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)
print(SOURCE, RUN_MODE, ARTIFACTS, HW)

## 2. Data

Besides the application, we use `bureau.csv`: the client's previous credits at other institutions, one row per credit. It is the closest thing to a "credit history" in the dataset.

In [ ]:
n_twin = int(307_511 * (SAMPLE_FRAC if RUN_MODE == "smoke" else 1))
app, bureau, DATASET = hc.load(SOURCE, DATA_DIR, n_synthetic=n_twin, seed=SEED)
if RUN_MODE == "smoke" and not OFFLINE:
    app = sample(app, SAMPLE_FRAC, SEED, stratify=hc.TARGET)
    bureau = bureau[bureau[hc.ID].isin(app[hc.ID])]
DATASET["sample_frac"] = SAMPLE_FRAC if RUN_MODE == "smoke" else 1.0
DATASET["n_rows"] = len(app)
print(app.shape, bureau.shape, f"default rate = {app[hc.TARGET].mean():.4f}")

## 3. Solution

### 3.1 One function, two consumers

The biggest risk of a production model is **training/serving skew**: features computed in the API differ from those used in training. That is why feature engineering lives in `cookbook.home_credit` and is called both here and in the API. The code is printed below for reading.

In [ ]:
for fn in (hc.engineer, hc.aggregate_bureau, hc.build_features):
    print(inspect.getsource(fn))

In [ ]:
with Timer() as t_feat:
    feats = hc.build_features(app, bureau)
feats["SPLIT"] = hc.split(feats[hc.ID]).to_numpy()
MODEL_COLUMNS = hc.model_columns(feats.drop(columns="SPLIT"))
print(feats.shape, f"{len(MODEL_COLUMNS)} model columns in {t_feat.seconds:.1f}s")
feats[["AGE_YEARS", "CREDIT_INCOME", "ANNUITY_INCOME", "EXT_SOURCE_MEAN", "BUREAU_COUNT", "BUREAU_DEBT_CREDIT_RATIO"]].describe().T

### 3.2 Data contract

Rules that, when broken, must **stop** the pipeline before training or deployment. In production the same contract runs on every load (see recipes K2.06 and B5.04).

In [ ]:
numeric = feats.select_dtypes("number")
ext = feats[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].to_numpy(dtype=float)
ext = ext[~np.isnan(ext)]
CONTRACT = {
    "unique applicant id": feats[hc.ID].is_unique,
    "binary target": set(feats[hc.TARGET].unique()) <= {0, 1},
    "protected attributes not in model": not set(hc.PROTECTED) & set(MODEL_COLUMNS),
    "no +/-inf values": not np.isinf(numeric.to_numpy(dtype=float)).any(),
    "external scores within [0, 1]": bool(((ext >= 0) & (ext <= 1)).all()),
    "age between 18 and 75": bool(feats["AGE_YEARS"].between(18, 75).all()),
    "bureau counts >= 0": bool((feats.filter(regex="^BUREAU_.*_COUNT$") >= 0).all().all()),
    "split shares ~60/20/20": bool(feats["SPLIT"].value_counts(normalize=True).between(0.15, 0.65).all()),
}
broken = [k for k, ok in CONTRACT.items() if not ok]
assert not broken, f"data contract broken: {broken}"
pd.Series(CONTRACT, name="ok")

### 3.3 Frozen category vocabulary

Categories seen in **training** become the official vocabulary. In the API, a new category becomes missing instead of shifting codes. We save the feature table and the metadata (columns, vocabulary, version).

In [ ]:
_, CATEGORIES = hc.to_categorical(feats.loc[feats["SPLIT"] == "train", MODEL_COLUMNS])
meta = {
    "model_columns": MODEL_COLUMNS,
    "categorical": [c for c in hc.CATEGORICAL if c in MODEL_COLUMNS],
    "categories": CATEGORIES,
    "dataset": DATASET,
    "run_mode": RUN_MODE,
}
feats.to_parquet(ARTIFACTS / "features.parquet", index=False)
(ARTIFACTS / "feature_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
{k: len(v) for k, v in CATEGORIES.items()}

## 4. Validation

Discriminating power of each numeric feature on its own (|Gini| on validation) and bureau coverage. It checks whether the engineering makes sense before training any model.

In [ ]:
valid = feats[feats["SPLIT"] == "valid"]
rows = []
for c in MODEL_COLUMNS:
    if c in CATEGORIES:
        continue
    m = valid[c].notna()
    if m.sum() > 100 and valid.loc[m, c].nunique() > 1:
        rows.append({"feature": c, "abs_gini": abs(cr.gini(valid.loc[m, hc.TARGET], valid.loc[m, c])),
                     "missing": 1 - m.mean()})
univariate = pd.DataFrame(rows).sort_values("abs_gini", ascending=False)
BUREAU_COVERAGE = float(feats["BUREAU_HAS_HISTORY"].mean())
print(f"bureau coverage: {BUREAU_COVERAGE:.1%}")
univariate.head(15)

## 5. Variation axes

### 5.1 With and without bureau features

A LightGBM with fixed parameters, trained with and without the `BUREAU_*` features, measures the value of the relational source before investing in integrating it.

In [ ]:
X_all, _ = hc.to_categorical(feats[MODEL_COLUMNS], CATEGORIES)
y_all = feats[hc.TARGET].to_numpy()
is_tr, is_va = (feats["SPLIT"] == "train").to_numpy(), (feats["SPLIT"] == "valid").to_numpy()
params = {"objective": "binary", "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 100,
          "feature_fraction": 0.8, "verbose": -1, "seed": SEED}

def quick_gini(columns):
    dtr = lgb.Dataset(X_all.loc[is_tr, columns], y_all[is_tr])
    dva = lgb.Dataset(X_all.loc[is_va, columns], y_all[is_va], reference=dtr)
    booster = lgb.train(params, dtr, 2000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return cr.bootstrap(cr.gini, y_all[is_va], booster.predict(X_all.loc[is_va, columns]), n=N_BOOT, seed=SEED)

no_bureau = [c for c in MODEL_COLUMNS if not c.startswith("BUREAU_")]
AXIS = {"with bureau": quick_gini(MODEL_COLUMNS), "without bureau": quick_gini(no_bureau)}
pd.DataFrame(AXIS).T

## 6. Pitfalls demonstrated

### 6.1 *Target encoding* computed before the split

**Symptom:** a high-cardinality category (`ORGANIZATION_TYPE`) looks far more predictive than it is. **Cause:** the per-category target mean included the test rows. **Fix:** compute encodings on training data only (with internal cross-validation), or let LightGBM handle the category natively, as we do.

In [ ]:
test = feats[feats["SPLIT"] == "test"]
train = feats[feats["SPLIT"] == "train"]
leaky_map = feats.groupby("ORGANIZATION_TYPE", dropna=False)[hc.TARGET].mean()      # WRONG: uses test labels
clean_map = train.groupby("ORGANIZATION_TYPE", dropna=False)[hc.TARGET].mean()
prior = train[hc.TARGET].mean()
enc = lambda m: test["ORGANIZATION_TYPE"].map(m).fillna(prior)
pd.Series({"encoding fit on train": cr.gini(test[hc.TARGET], enc(clean_map)),
           "encoding fit on all rows (leak)": cr.gini(test[hc.TARGET], enc(leaky_map))}, name="test gini")

## 7. Results

In [ ]:
for variant, gini_value in AXIS.items():
    save_result(
        recipe_id="P1.01", notebook="30-feature-pipeline", stack="lightgbm-fixed-params",
        variant=variant, run_mode=RUN_MODE, hardware=HW, versions=VERSIONS, dataset=DATASET,
        params={"seed": SEED, **params}, metrics={"gini_valid": gini_value},
        time_s={"train": t_feat.seconds, "inference_per_1k": None},
    )
save_result(
    recipe_id="P1.01", notebook="30-feature-pipeline", stack="pandas+polars",
    run_mode=RUN_MODE, hardware=HW, versions=VERSIONS, dataset=DATASET, params={"seed": SEED},
    metrics={
        "n_features": {"mean": len(MODEL_COLUMNS), "std": None, "n": 1},
        "bureau_coverage": {"mean": round(BUREAU_COVERAGE, 6), "std": None, "n": 1},
        "contract_checks_passed": {"mean": sum(CONTRACT.values()), "std": None, "n": 1},
    },
    time_s={"train": t_feat.seconds, "inference_per_1k": round(t_feat.seconds / len(feats) * 1000, 6)},
)

## 8. Conclusion

- Application + bureau features, a data contract that blocks bad loads and a frozen vocabulary.
- The bureau gain shows up in section 5: it decides whether integrating this source in production pays off.
- **Next step:** [31 · calibrated monotonic GBM](31-gbm-monotonic-calibrated.ipynb).